# A. Data

In [ ]:
import pandas as pd

In [ ]:
file_path = "listings_newyork.xlsx"
listing = pd.read_excel(file_path)
listing.head()

In [ ]:
listing.columns

In [ ]:
listing.shape

# B. Preproccesing

## 1. Data cleansing and transformation

### 1.1. Drop meaningless columns

Dropping columns that have no predictive signal/impossible to transform:
1. Pure identification/URL columns: 'id', 'listing_url', 'scrape_id', 'host_id', 'host_url', 'picture_url', 'host_thumbnail_url', 'host_picture_url'
2. Long unstructured text which are not relevant to the prediction: 'name', 'description', 'host_name', 'host_since', 'host_about', 'neighborhood_overview', 'host_about', 'host_verifications', 'source'
4. High-cardinality, overly specific location fields: 'host_location',
'host_neighbourhood', 'neighbourhood',  'neighbourhood_cleansed' (we keep only 'neighbourhood_group_cleansed' as it shows the aggregate data)
5. Redundant/overlapping columns (as they are sub-components of the same thing):
   * 'property_type' --> keep 'room_type' only (because room_type is more representative, showing the types of property)
   * 'host_listings_count' --> keep 'host_total_listings_count' only
6. Columns with extremely high missingness: 'license'
7. Machine-generated timing fields (these are internal system timestamps → irrelevant to Superhost prediction): 'calendar_updated', 'calendar_last_scraped', 'last_scraped' (we may use 'last_scraped' together with 'first_review' and 'last_review' to calculate the total time the listings have been reviewd on Airbnb, then we drop those columns later)

In [ ]:
listing["property_type"].value_counts()

In [ ]:
listing["calculated_host_listings_count_entire_homes"].isnull().sum()

In [ ]:
listing["calculated_host_listings_count_private_rooms"].isnull().sum()

In [ ]:
listing["calculated_host_listings_count_shared_rooms"].isnull().sum()

In [ ]:
listing.drop(columns=[
    'id', 'listing_url', 'scrape_id', 'host_id', 'host_url', 'picture_url', 'host_thumbnail_url', 'host_picture_url',
    'name', 'description', 'host_name', 'neighborhood_overview', 'host_verifications', 'source',
    'host_location', 'host_neighbourhood', 'neighbourhood', 'neighbourhood_cleansed',
    'property_type', 'host_listings_count', 'license', 'calendar_updated', 'calendar_last_scraped', 'host_since', 'host_about'
], inplace=True)   

In [ ]:
listing.columns

In [ ]:
listing.shape

### 1.2. Data transformations

#### a. Convert 'amenities' to 'amenities_count'

This will be better instead of dropping this column out of our dataset because
* The number of amenities strongly reflects listing quality --> affects guest satisfaction --> affects review scores --> affects chances of being Superhost.

In [ ]:
listing["amenities"]

In [ ]:
import ast
import pandas as pd

def count_amenities(x):
    # Case 1: already a Python list
    if isinstance(x, list):
        return len(x)
    
    # Case 2: it's a string
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return 0
        try:
            # Try to interpret it as a Python-style list string
            items = ast.literal_eval(s)
            return len(items)
        except Exception:
            # Fallback: treat it as a comma-separated string
            return len([a for a in s.split(',') if a.strip() != ""])
    
    # Case 3: NaN or anything else
    return 0

listing['amenities_count'] = listing['amenities'].apply(count_amenities)

In [ ]:
listing["amenities_count"].head()

In [ ]:
listing.drop(columns=['amenities'], inplace=True)

#### b. Convert data variables --> numeric features

##### *review recency 

* last_scraped - first_review = how long it has been since the host got their first review
  
--> Transform to 'days_since_first_rv'

* last_scraped - last_review = how long since their most recent review
  
--> Transform to 'days_since_last_rv'

These can be strong predictors because:
* Superhosts must have consistent bookings
* If last review is long ago → low activity → less likely Superhost

In [ ]:
#convert to datetime
listing["first_review"] = pd.to_datetime(listing["first_review"], errors = 'coerce')
listing["last_review"] = pd.to_datetime(listing["last_review"], errors = 'coerce')

In [ ]:
#days since first review
listing["days_since_first_rv"] = (listing['last_scraped'] - listing['first_review']).dt.days
#days since last review
listing['days_since_last_rv'] = (listing['last_scraped'] - listing['last_review']).dt.days

In [ ]:
#drop the original date columns
listing.drop(columns=["last_scraped", "first_review", "last_review"], inplace=True)

In [ ]:
listing.head()

In [ ]:
listing.columns

#### d. Categorical variables

##### *bathrooms_text vs. bathrooms

In [ ]:
listing["bathrooms_text"].isnull().sum()

In [ ]:
listing["bathrooms"].isnull().sum()

In [ ]:
listing["bathrooms_text"].value_counts()

In [ ]:
mapping1 = {
    "0 baths": 0,
    "0 shared baths": 0,
    "Half-bath": 0.5,
    "Shared half-bath": 0.5,
    "Private half-bath": 0.5,

    "1 bath": 1,
    "1 shared bath": 1,
    "1 private bath": 1,

    "1.5 baths": 1.5,
    "1.5 shared baths": 1.5,

    "2 baths": 2,
    "2 shared baths": 2,
    "2.5 baths": 2.5,
    "2.5 shared baths": 2.5,

    "3 baths": 3,
    "3 shared baths": 3,
    "3.5 baths": 3.5,
    "3.5 shared baths": 3.5,

    "4 baths": 4,
    "4 shared baths": 4,
    "4.5 baths": 4.5,

    "5 baths": 5,
    "5 shared baths": 5,
    "5.5 baths": 5.5,

    "6 baths": 6,
    "6 shared baths": 6,
    "6.5 baths": 6.5,

    "7 baths": 7,
    "7.5 baths": 7.5,

    "9 baths": 9,
    "10.5 baths": 10.5,
    "15.5 baths": 15.5
}

In [ ]:
listing["bathrooms_text"] = listing["bathrooms_text"].map(mapping1)
listing["bathrooms_text"].value_counts()

In [ ]:
listing["bathrooms_number"] = listing["bathrooms_text"]
listing.drop(columns = ["bathrooms_text"], inplace = True) #already renamed bathrooms_text to bathrooms

In [ ]:
listing.drop(columns = ["bathrooms"], inplace = True)

##### *Ordinal encoding host_response_time

In [ ]:
listing["host_response_time"]

Ordinal encoding makes sense because response time is ordered:
* Fast = Good
* Slow = Bad

In [ ]:
#Ordinal Encoding (fastest response = highest score)
mapping2 = {
    'within an hour': 4,
    'within a few hours': 3,
    'within a day': 2,
    'a few days or more': 1
}

listing['host_response_time_score'] = listing['host_response_time'].map(mapping2)

#Drop the original host_response_time column
listing.drop(columns = ["host_response_time"], inplace = True)
listing['host_response_time_score'].describe()

##### *Convert t/f to binary variables (1/0): super_host, instant_bookable, has_availability, host_has_picture_pic, host_identity_verified

In [ ]:
listing["host_is_superhost"].value_counts()

In [ ]:
listing["instant_bookable"].value_counts()

In [ ]:
listing["has_availability"].value_counts()

In [ ]:
listing["has_availability"].isnull().sum()

In [ ]:
listing["host_has_profile_pic"].value_counts()

In [ ]:
listing["host_identity_verified"].value_counts()

In [ ]:
mapping3 = {"t":1, "f":0}
listing["host_is_superhost"] = listing["host_is_superhost"].map(mapping3)
listing["instant_bookable"] = listing["instant_bookable"].map(mapping3)
listing["has_availability"] = listing["has_availability"].map(mapping3).fillna(0) 
listing["host_has_profile_pic"] = listing["host_has_profile_pic"].map(mapping3)
listing["host_identity_verified"] = listing["host_identity_verified"].map(mapping3)
#we filled 0 to missing values in has_availability as this column shows only either "t" or "NaaN"

##### *Transfrom neighbourhood_group_cleansed and room_type to dummies

In [ ]:
listing["neighbourhood_group_cleansed"].value_counts()

In [ ]:
listing["neighbourhood_group"] = listing["neighbourhood_group_cleansed"]
listing.drop(columns = ["neighbourhood_group_cleansed"], inplace = True)
listing["neighbourhood_group"]

In [ ]:
listing["room_type"].value_counts()

In [ ]:
listing = pd.get_dummies(listing, columns = ["neighbourhood_group", "room_type"], drop_first = True, dtype = int)

##### *Convert % variables to numeric: host_response rate, host_acceptance_rate

In [ ]:
listing['host_response_rate'] = (
    listing['host_response_rate']
        .astype(str)          # convert all to string
        .str.replace('%', '') # remove %
        .replace('nan', None) # convert "nan" strings back to real NaN
        .astype(float)        # convert to float
)
listing['host_response_rate'] = listing['host_response_rate']

In [ ]:
listing['host_acceptance_rate'] = (
    listing['host_acceptance_rate']
        .astype(str)          # convert all to string
        .str.replace('%', '') # remove %
        .replace('nan', None) # convert "nan" strings back to real NaN
        .astype(float)        # convert to float
)
listing['host_acceptance_rate'] = listing['host_response_rate']

### 1.3. Handle missing values

In [ ]:
listing.isnull().sum()

In [ ]:
listing.shape

#### a. Columns with small numbers of missing values: Listwise deletion

In [ ]:
listing.dropna(subset=['host_is_superhost', 'host_total_listings_count', 'host_has_profile_pic', 'host_identity_verified', 
                       'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights',
                       'bathrooms_number'
                      ], inplace = True)  

In [ ]:
listing.isnull().sum()

#### b. Host-related

In [ ]:
listing.columns

In [ ]:
host_cols = [
    'host_response_rate', 'host_acceptance_rate'
    ]

listing[host_cols].describe()

In [ ]:
from matplotlib import pyplot as plt
listing[host_cols].hist(bins=50, figsize=(7, 4))
plt.tight_layout()
plt.show()

In [ ]:
review_cols = [
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_checkin',
    'review_scores_communication', 'review_scores_location',
    'review_scores_value'
]

listing[review_cols].describe()

In [ ]:
listing[review_cols].hist(bins=50, figsize=(12, 10))
plt.tight_layout()
plt.show()

In [ ]:
#mean substitution
for col in host_cols:
    listing[col] = listing[col].fillna(listing[col].mean())

In [ ]:
#mean substitution
for col in review_cols:
    listing[col] = listing[col].fillna(listing[col].mean())

#### c. Property attributes

* price: Numeric variables with highly skewed distributions --> replace missing price values using the mean, but only using values within the 3rd quartile (≤ 75th percentile).
* beds: Use mode
* bathrooms_number: Use mode

In [ ]:
listing["price"].describe() #very highly skewed 

In [ ]:
# 1. Compute Q3 (75th percentile)
q3 = listing["price"].quantile(0.75)

# 2. Compute mean within Q3 (mean of lower 75% of prices)
mean_within_q3 = listing[listing["price"] <= q3]["price"].mean()

# 3. Replace missing values using this trimmed mean
listing["price"] = listing["price"].fillna(mean_within_q3)

In [ ]:
bedroom_mode = listing["bedrooms"].mode() [0]
bedroom_mode

In [ ]:
bed_mode = listing["beds"].mode() [0]
bed_mode

In [ ]:
bathrooms_number_mode = listing["bathrooms_number"].mode() [0]
bathrooms_number_mode

In [ ]:
listing.fillna({"bathrooms_number":bathrooms_number_mode, "bedrooms":bedroom_mode, "beds":bed_mode}, inplace = True)

In [ ]:
listing.isnull().sum()

In [ ]:
listing.shape

#### d. Behavior-related data

* estimated_revenue_l365d (>14,000 missing)
--> Missing means listing had zero bookings --> behavior signal --> replace by 0 instead of using mean or median

In [ ]:
listing['estimated_revenue_l365d'].value_counts()

In [ ]:
listing['estimated_revenue_l365d'] = listing['estimated_revenue_l365d'].fillna(0)

* reviews_per_month --> Missing means listing has no reviews --> better replace by 0

In [ ]:
listing['reviews_per_month'].value_counts()

In [ ]:
listing["reviews_per_month"].describe()

In [ ]:
listing['reviews_per_month'] = listing['reviews_per_month'].fillna(0)

In [ ]:
listing["host_response_time_score"].value_counts()

In Airbnb, missing response-time metrics typically indicate hosts who have not previously responded or are inactive. Therefore, missing values were imputed with the lowest response-time score (1), representing the slowest response behavior. This approach preserves business meaning and avoids overestimating host responsiveness.

In [ ]:
listing["host_response_time_score"] = listing["host_response_time_score"].fillna(1.0)

In [ ]:
listing.isnull().sum()

In [ ]:
listing["minimum_minimum_nights"].describe()

In [ ]:
listing["maximum_minimum_nights"].describe()

In [ ]:
listing["minimum_maximum_nights"].describe()

In [ ]:
listing["maximum_maximum_nights"].describe()

In [ ]:
listing.dropna(subset = ["minimum_minimum_nights", "maximum_minimum_nights", "minimum_maximum_nights", "maximum_maximum_nights"], inplace = True)

In [ ]:
listing.isnull().sum()

#### e. Review recency variables (days_since_first_rv, days_since_last_rv) --> replace by meaningful value

In [ ]:
listing["days_since_first_rv"].describe()

In [ ]:
listing["days_since_last_rv"].describe()

NOTE: days_since_first_rv, days_since_last_rv

--> Missing = listing never reviewed (not just simply mean that there are missing values) 

--> This is a strong negative signal --> replace by an extremly large value (9999)

In [ ]:
listing['days_since_first_rv'] = listing['days_since_first_rv'].fillna(9999)
listing['days_since_last_rv'] = listing['days_since_last_rv'].fillna(9999)

### *Final check

In [ ]:
listing.isnull().sum()

## 2. Splitting dataset and Scaling

### 2.1 Arrange the columns

In [ ]:
cols = listing.columns.tolist()
cols

In [ ]:
listing.head()

In [ ]:
listing.shape

In [ ]:
listing.columns.get_loc("latitude")

In [ ]:
listing.columns.get_loc("neighbourhood_group_Brooklyn")

In [ ]:
cols1 = [cols[2]] + cols[0:2] + cols[3:6] + cols[8:50] + cols[54:] + cols[50:54] + cols[6:8]
print(cols1)

In [ ]:
listing1 = listing[cols1]
listing1.shape

In [ ]:
y = listing1.iloc[:, 0] #host_is_super_host
x = listing1.iloc[:,1:51] #all other columns

In [ ]:
y.name

In [ ]:
x.columns

In [ ]:
x.shape

### 2.2. Splitting to train set and test set

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state=0) 

### 2.3. Data resampling (only the training set)

In [ ]:
from sklearn.utils import resample

In [ ]:
y_train.value_counts()

In [ ]:
train0 = x_train[y_train == 0].copy() #non-superhost

In [ ]:
train1 = x_train[y_train == 1].copy() #superhost

In [ ]:
train0_downsample = resample(train0, replace=True, n_samples=len(train1), random_state=0) #downsample majority class

In [ ]:
x_train_resampled = pd.concat([train1, train0_downsample], axis=0) #concatenate back x_trained

In [ ]:
#build matching y for the resampled x
y0 = y_train[y_train == 0]
y1 = y_train[y_train == 1]
y0_downsample = resample(y0, replace=True, n_samples=len(y1), random_state=0)
y_train_resampled = pd.concat([y1, y0_downsample], axis=0)

In [ ]:
y_train_resampled.value_counts()

### 2.4. Scaling the data (fit only the training set)

In [ ]:
from sklearn.preprocessing import StandardScaler #scaled data will be used for SVM, KNN, MLP

In [ ]:
scaler = StandardScaler() #build scaler object

In [ ]:
scaler_model = scaler.fit(x_train_resampled) #fit on the training data only

In [ ]:
x_train_scaled = scaler_model.transform(x_train_resampled) #transform training set
x_test_scaled = scaler_model.transform(x_test) #transform test set